# Calificador de casas — California Housing (por fases)
Fases 1-5: carga → K-Means k=6 → test → clasificador supervisado → guardado. Ver `docs/` para utilidad y tech de cada fase.

In [ ]:
import sys
sys.path.insert(0, '.')
from data_loader import load_housing, split_data
from clustering import fit_kmeans, predict_clusters
from classification import features_and_target, train_and_evaluate
from visualization import scatter_clusters, scatter_train_test

df = load_housing('../data/raw/housing.csv')
train, test = split_data(df)
print(f'total={df.shape} train={train.shape} test={test.shape}')
train.describe()

## Fase 2 — K-Means k=6 (ver `docs/FASE_02_kmeans.md`)

In [ ]:
scaler, kmeans, train_labeled = fit_kmeans(train)
print(kmeans.cluster_centers_)
print(train_labeled['cluster'].value_counts().sort_index())
scatter_clusters(train_labeled, 'Clusters K-Means (k=6) sobre train', '../reports/figures/clusters_train.png')

## Fase 3 — Predicción en test (ver `docs/FASE_03_prediccion_test.md`)

In [ ]:
test_labeled = predict_clusters(scaler, kmeans, test)
print(test_labeled['cluster'].value_counts().sort_index())
scatter_train_test(train_labeled, test_labeled, '../reports/figures/clusters_train_test.png')

## Fase 4 — Clasificación supervisada (ver `docs/FASE_04_clasificacion_supervisada.md`)

In [ ]:
Xtr_raw, ytr = features_and_target(train_labeled)
Xte_raw, yte = features_and_target(test_labeled)
Xtr, Xte = scaler.transform(Xtr_raw), scaler.transform(Xte_raw)
best_name, best_model, results = train_and_evaluate(Xtr, ytr, Xte, yte)
print({k: v['accuracy'] for k, v in results.items()}, 'mejor:', best_name)
print(results[best_name]['report_text'])

## Fase 5 — Guardado (ver `docs/FASE_05_guardado_modelos.md`)

In [ ]:
import joblib
joblib.dump(kmeans, '../models/kmeans.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(best_model, '../models/classifier.pkl')
print('modelos guardados en models/')